# REINFORCE on the continuous mountain car

A port of `reinforce_mountaincar.py` from
[DuarteA/ActorCriticExample](https://github.com/DuarteA/ActorCriticExample),
the repository the deck links to on slide 19.

REINFORCE is the Monte-Carlo policy gradient method of this unit: run an
episode under the current stochastic policy, compute the return from every
step, and move the parameters in the direction

$$\theta_{k+1}=\theta_k-\gamma\,G_k\,\nabla_\theta\log\pi(u_k\mid x_k,\theta_k).$$

Nothing is estimated except the policy. There is no cost-to-go, no Q-factor
and no table -- which is what the deck means by *approximation in policy
space*.

::: Requirements :::
`torch` and `gymnasium` (classic control only -- no Box2D needed):

``` bash
pip install torch "gymnasium[classic_control]"
```

The agents live in `policy_gradient.py` next to this notebook.

**Sign convention.** Gymnasium hands out *rewards to be maximised*; Units
1--12 minimise a *cost*. The deck's descent on a cost and this code's descent
on the loss $-\log\pi\cdot R$ are the same thing with $g=-R$.

In [ ]:
import sys, time
import numpy as np
import torch
import matplotlib.pyplot as plt

# policy_gradient.py sits next to this notebook -- download it alongside
sys.path.insert(0, ".")
import policy_gradient as pg
from policy_gradient import Config, seed_all, make_env, evaluate, smooth

EPISODES = 1000      # the script's default; 200-300 already shows the behaviour
cfg = Config(episodes=EPISODES)
seed_all(cfg.seed)
print(f"environment {cfg.env_id}, discount {cfg.gamma}")

## The environment

A car is placed stochastically at the bottom of a sinusoidal valley. The only
actions are accelerations applied in either direction, and the engine is **not
strong enough to drive straight up the hill** -- the car has to rock back and
forth to build momentum.

| | |
|:--|:--|
| state | position and velocity, both continuous |
| action | acceleration in $[-1,1]$, continuous |
| reward | $-0.1\,u^2$ at every step, $+100$ on reaching the goal |

The action space is continuous, which is the reason this example belongs to
this unit rather than to Unit 12: a Q-factor method would have to minimise
over a continuous set at every step, while a policy gradient method simply
outputs the action.

Note the shape of the reward, because the whole notebook turns on it. Every
unit of acceleration costs something, and the $+100$ arrives only if the car
reaches the flag. A policy that does **nothing** scores about $0$. A policy
that burns fuel without reaching the goal scores perhaps $-50$. So doing
nothing is better than trying and failing, and the agent has to discover the
goal *before* it learns to stop wasting fuel.

## The policy

The policy must be **stochastic**, for the reason the unit opens with: over a
finite action set a deterministic policy makes the cost a discontinuous
function of the parameters, and there is nothing to differentiate. Here the
action set is continuous, and the natural parameterisation is a **Gaussian**:
the network maps the state to a mean and a log standard deviation, and the
action is drawn from that normal distribution.

So $\theta$ is the network's weights, and $\pi(u\mid x,\theta)$ is a normal
density -- differentiable in $\theta$, which is all the policy gradient
theorem asks for.

In [ ]:
env = make_env(cfg, seed=cfg.seed)
state_size = env.observation_space.shape[0]
action_size = env.action_space.shape[0]
agent = pg.REINFORCE(state_size, action_size, cfg)
print(agent.policy)
print("policy parameters:", sum(p.numel() for p in agent.policy.parameters()))

# what the untrained policy looks like at the bottom of the valley
with torch.no_grad():
    mean, log_std = agent.policy(torch.tensor([[-0.5, 0.0]]))
print(f"\nat rest at the valley floor: action ~ Normal(mean={float(mean):+.3f}, "
      f"std={float(torch.exp(log_std)):.3f})")

## One detail in the update that is worth naming

The script normalises the returns before forming the loss,

$$\tilde G_k=\frac{G_k-\bar G}{\text{std}(G)+\varepsilon}.$$

Subtracting the mean is exactly the **baseline subtraction** of the deck's
slide 13: a term $b(x)$ that does not depend on the action leaves the gradient
unchanged in expectation, because
$\sum_u\nabla_\theta\pi(u\mid x,\theta)=\nabla_\theta(1)=0$. So this
"REINFORCE" is already REINFORCE *with a baseline* -- a crude, constant one.

Dividing by the standard deviation is not a baseline and does change the
update: it rescales the step size episode by episode. It is a practical
stabiliser, not something the theory asks for.

Both matter for the comparison with actor-critic in the next notebook: the
actor-critic's advantage is a *state-dependent* learned baseline, and the gain
over this constant one is the thing being measured. `cfg.normalise_returns`
turns it off if you want to see the raw method.

## Training

One update per episode. Expect several minutes.

In [ ]:
t0 = time.perf_counter()
scores, losses, return_std = pg.train_reinforce(agent, env, cfg, verbose_every=100)
print(f"\n{len(scores)} episodes in {time.perf_counter()-t0:.0f} s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(scores, alpha=0.3, lw=1, color="C0")
sm = smooth(scores, 50)
ax[0].plot(np.arange(len(sm)) + 49, sm, color="C3", lw=2, label="moving average (50)")
ax[0].axhline(0, color="k", ls=":", lw=1)
ax[0].axhline(90, color="g", ls="--", lw=1, label="reaching the goal")
ax[0].set_xlabel("Episode"); ax[0].set_ylabel("Episode return"); ax[0].legend()
ax[0].set_title("REINFORCE on MountainCarContinuous"); ax[0].grid(True, alpha=0.3)

ax[1].plot(return_std, lw=1, color="C2")
ax[1].set_xlabel("Episode"); ax[1].set_ylabel("std of the returns within an episode")
ax[1].set_title("Spread of $G_k$ -- the quantity the gradient is weighted by")
ax[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
print(f"best episode          {max(scores):8.2f}")
print(f"mean of last 100      {np.mean(scores[-100:]):8.2f}")
print(f"episodes reaching the goal (score > 90): {sum(s > 90 for s in scores)} of {len(scores)}")
with torch.no_grad():
    _, log_std = agent.policy(torch.tensor([[-0.5, 0.0]]))
print(f"final policy std at the valley floor: {float(torch.exp(log_std)):.4f}")

## Reading the result

Do not expect this to solve the problem, and the reason is worth more than a
success would be.

**The "do nothing" trap.** Look at where the score settles. Acting costs
$-0.1u^2$ per step and the $+100$ only arrives at the flag, so until the car
has *reached the flag at least once* the gradient has no information that
acting is ever worthwhile -- it only sees fuel being burned. The policy
therefore improves by shrinking its actions towards zero, the score climbs
towards $0$ from below, and the standard deviation of the Gaussian collapses.
Once it has collapsed, the policy stops exploring and can never discover the
goal. This is a local optimum, and it is the one REINFORCE reliably finds
here.

**This is an exploration failure, not a bug.** The deck's "cons" slide lists
high variance and sensitivity to learning rates. The mountain car adds the
third problem that policy gradient methods have and value-based methods
handle better: a policy gradient method explores only through the randomness
*of its own policy*, and that randomness is itself a learned parameter which
the objective is actively pushing to zero. There is no $\epsilon$-greedy
floor here as there was in Units 11 and 12.

**What a fix looks like.** Anything that keeps the policy exploring until the
goal is found: an entropy bonus added to the objective (this is what Soft
Actor-Critic on the deck's variants slide is for), a lower bound on the log
standard deviation, reward shaping that credits building momentum, or simply a
luckier seed. The cell below tries the cheapest of them -- holding the
exploration open with a floor on the standard deviation -- and is worth
running if the main training settled at zero.

In [ ]:
# Optional: the same run with the policy's exploration held open.
# Clamping log_std from below stops the Gaussian from collapsing before the
# car has ever reached the flag.
RUN_FLOOR_EXPERIMENT = True
FLOOR_EPISODES = min(300, EPISODES)

if RUN_FLOOR_EXPERIMENT:
    import torch.nn.functional as F
    seed_all(cfg.seed)
    env2 = make_env(cfg, seed=cfg.seed)
    agent2 = pg.REINFORCE(state_size, action_size, cfg)
    original_forward = agent2.policy.forward

    def floored_forward(state, _f=original_forward):
        mean, log_std = _f(state)
        return mean, torch.clamp(log_std, min=np.log(0.5))   # std never below 0.5

    agent2.policy.forward = floored_forward
    s2, _, _ = pg.train_reinforce(agent2, env2, cfg, episodes=FLOOR_EPISODES, verbose_every=100)
    print(f"\nwith an exploration floor: best {max(s2):.2f}, "
          f"last50 mean {np.mean(s2[-50:]):.2f}, "
          f"episodes past the flag: {sum(x > 90 for x in s2)}")
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(s2, alpha=0.35, lw=1, color="C1")
    sm2 = smooth(s2, 25)
    ax.plot(np.arange(len(sm2)) + 24, sm2, color="C1", lw=2)
    ax.axhline(90, color="g", ls="--", lw=1)
    ax.set_xlabel("Episode"); ax.set_ylabel("Episode return")
    ax.set_title("REINFORCE with the policy standard deviation floored at 0.5")
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

Whether that rescues the run depends on the seed, which is itself the
honest lesson: a method whose only exploration is the width of its own output
distribution is fragile on a problem with a sparse reward. The next notebook
adds a critic, which reduces the variance of the gradient but does **not** fix
this -- the trap is in the objective, not in the estimator.